## Ewaluacja RAG

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mzarnecki/course_llm_agent_apps_with_langchain_and_langgraph/blob/main/notebooks/pl/7_6_RAG_evaluation.ipynb)

| | |
|---|---|
| 🎯 **Czego się nauczysz** | Jak sprawdzić, czy aplikacja RAG naprawdę działa dobrze: metryki wyszukiwania (Recall@K, Precision@K, MRR), LLM jako sędzia dla wierności i poprawności odpowiedzi, trafność cytowań, opóźnienie i zużycie tokenów na małym złotym zbiorze testowym. |
| 🔑 **Klucze API** | `OPENAI_API_KEY` |
| 💲 **Koszt API** | niski – ok. 50 krótkich zapytań |
| ⏱️ **Czas wykonania** | ~2 min |

### Instalacja bibliotek

In [1]:
# Zainstaluj bezpośrednie zależności używane w tym notebooku.
# Pomijane, gdy brak pip, np. w środowisku `uv sync`, które ma je już zainstalowane (zob. README).
import importlib.util
if importlib.util.find_spec("pip"):
    %pip install -qU langchain-openai pandas python-dotenv
# Google Colab: dodaj OPENAI_API_KEY w panelu Secrets (🔑) i zezwól notatnikowi na dostęp.
# Odkomentuj te linie, aby wczytać klucz z Colab Secrets do bieżącej sesji:
# from google.colab import userdata
# import os
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# Lokalnie umieść OPENAI_API_KEY=... w pliku .env; nie commituj klucza.

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

### Co mierzymy?

Aplikacja RAG może zawieść w dwóch miejscach: **retriever** może nie znaleźć właściwych fragmentów, a **LLM** może źle odpowiedzieć nawet przy dobrym kontekście. Dlatego oba etapy mierzymy osobno, a do tego koszt całego pipeline'u.

| Etap | Metryka | Co sprawdzamy? |
|---|---|---|
| Wyszukiwanie | **Recall@K** | Czy znaleziono potrzebne fragmenty w top K? |
| Wyszukiwanie | **Precision@K** | Ile znalezionych fragmentów jest istotnych? |
| Wyszukiwanie | **MRR** | Jak wysoko znalazł się pierwszy właściwy wynik? |
| Generowanie | **Faithfulness** | Czy odpowiedź wynika z dostarczonych źródeł (brak halucynacji)? |
| Generowanie | **Answer correctness** | Czy odpowiedź jest poprawna względem wzorca? |
| Generowanie | **Citation accuracy** | Czy wskazano odpowiednie źródła? |
| Koszt | **Latency** | Ile czasu zajęło wygenerowanie odpowiedzi? |
| Koszt | **Token usage** | Ile zasobów zużyła aplikacja? |

Na start nie potrzeba tysięcy przykładów: **10–20 pytań z oczekiwanymi odpowiedziami i identyfikatorami właściwych dokumentów** wystarczy, żeby wyłapać większość regresji.

### 1. Baza wiedzy z identyfikatorami dokumentów
Stałe identyfikatory to podstawa ewaluacji – bez nich nie da się powiedzieć, *który* dokument powinien zostać znaleziony. Tutaj każdy artykuł to jeden chunk; przy dzieleniu tekstu nadaj każdemu chunkowi ID dokumentu źródłowego (np. w `metadata`).

In [3]:
from langchain_core.documents import Document

# Artykuły z centrum pomocy fikcyjnego produktu SaaS "Orbitly".
# Każdy artykuł ma stały identyfikator - zbiór testowy odwołuje się do tych ID.
KNOWLEDGE_BASE = {
    "plans-01": "Orbitly oferuje trzy plany. Free: do 3 projektów i 1 GB miejsca. Pro: 49 zł za użytkownika miesięcznie, nielimitowane projekty i 100 GB miejsca. Enterprise: wycena indywidualna, SSO i dedykowany opiekun klienta.",
    "billing-01": "Faktury są wystawiane 1. dnia każdego miesiąca i wysyłane na adres e-mail do rozliczeń. Klienci planu Pro płacą kartą kredytową. Klienci Enterprise mogą też płacić przelewem z 30-dniowym terminem płatności.",
    "billing-02": "Za plany roczne można otrzymać zwrot pieniędzy w ciągu 14 dni od zakupu. Plany miesięczne nie podlegają zwrotowi, ale anulowanie planu miesięcznego wstrzymuje kolejne obciążenie.",
    "trial-01": "Wersja próbna planu Pro trwa 21 dni i nie wymaga karty kredytowej. Po jej zakończeniu przestrzeń robocza automatycznie przechodzi na plan Free, a dane ponad limity planu Free stają się tylko do odczytu.",
    "security-01": "Wszystkie dane są szyfrowane w spoczynku algorytmem AES-256, a w trakcie przesyłania protokołem TLS 1.3. Dane są przechowywane we Frankfurcie (UE) albo w Wirginii (USA); region wybiera się przy tworzeniu przestrzeni roboczej.",
    "security-02": "Uwierzytelnianie dwuskładnikowe (2FA) działa z aplikacją uwierzytelniającą lub sprzętowym kluczem bezpieczeństwa (WebAuthn). Administratorzy planów Pro i Enterprise mogą wymusić 2FA dla wszystkich członków przestrzeni roboczej.",
    "sso-01": "Logowanie jednokrotne (SSO) przez SAML 2.0 i OIDC jest dostępne wyłącznie w planie Enterprise. Obsługiwani dostawcy tożsamości to Okta, Microsoft Entra ID (Azure AD) i Google Workspace.",
    "api-01": "REST API pozwala na 600 zapytań na minutę na przestrzeń roboczą w planie Pro i 3000 w planie Enterprise. Po przekroczeniu limitu API zwraca HTTP 429 z nagłówkiem Retry-After.",
    "api-02": "Klucze API tworzy się w Ustawienia -> Deweloper. Klucz jest wyświetlany tylko raz, może mieć zakres tylko do odczytu lub odczyt-zapis i wygasa po 365 dniach.",
    "export-01": "Pojedynczy projekt można wyeksportować do JSON, CSV lub PDF w menu Projekt -> Eksport. Administratorzy mogą zlecić pełny eksport przestrzeni roboczej jako plik ZIP; jest gotowy w ciągu 24 godzin, a link do pobrania jest ważny 7 dni.",
    "support-01": "Użytkownicy planu Free otrzymują pomoc na forum społeczności. Użytkownicy Pro mają wsparcie e-mail z pierwszą odpowiedzią w ciągu 24 godzin roboczych. Użytkownicy Enterprise mają wsparcie telefoniczne 24/7 i czas reakcji 1 godziny przy krytycznych incydentach.",
    "retention-01": "Usunięte projekty trafiają do Kosza na 30 dni, a potem są trwale usuwane. Kopie zapasowe są przechowywane 35 dni. Po zamknięciu konta wszystkie dane są usuwane w ciągu 60 dni.",
    "integrations-01": "Orbitly integruje się ze Slackiem, GitHubem, Google Drive i Jirą. Webhooki są dostępne w planach Pro i Enterprise; nieudane dostarczenia są ponawiane do 5 razy z wykładniczym opóźnieniem (exponential backoff).",
    "mobile-01": "Aplikacje mobilne są dostępne na iOS 16+ i Android 10+. W trybie offline zmiany są synchronizowane po odzyskaniu połączenia; konflikty edycji są rozwiązywane przez zachowanie obu wersji.",
}

docs = [Document(page_content=text, metadata={"id": doc_id}) for doc_id, text in KNOWLEDGE_BASE.items()]
print(f"Liczba dokumentów w bazie wiedzy: {len(docs)}")

Liczba dokumentów w bazie wiedzy: 14


### 2. Złoty zbiór testowy
15 pytań sformułowanych tak, jak zadają je prawdziwi użytkownicy (nie przepisanych z dokumentów). Trzy z nich wymagają **dwóch** dokumentów – sprawdzają, czy retriever zbiera kompletny kontekst.

In [4]:
# Złoty zbiór testowy: pytanie, odpowiedź wzorcowa, ID dokumentów zawierających odpowiedź.
EVAL_DATASET = [
    {"question": "Ile kosztuje plan Pro?",
     "reference": "49 zł za użytkownika miesięcznie.",
     "relevant_ids": ["plans-01"]},
    {"question": "Czy mogę dostać zwrot pieniędzy za plan miesięczny?",
     "reference": "Nie, plany miesięczne nie podlegają zwrotowi; anulowanie wstrzymuje tylko kolejne obciążenie.",
     "relevant_ids": ["billing-02"]},
    {"question": "Jak długo trwa okres próbny i czy potrzebuję karty kredytowej?",
     "reference": "Wersja próbna Pro trwa 21 dni i nie wymaga karty kredytowej.",
     "relevant_ids": ["trial-01"]},
    {"question": "Co się stanie z moją przestrzenią roboczą po zakończeniu okresu próbnego?",
     "reference": "Przejdzie na plan Free (3 projekty, 1 GB); dane ponad limity planu Free staną się tylko do odczytu.",
     "relevant_ids": ["trial-01", "plans-01"]},
    {"question": "Których dostawców tożsamości mogę użyć do SSO?",
     "reference": "Okta, Microsoft Entra ID (Azure AD) i Google Workspace, przez SAML 2.0 lub OIDC, tylko w planie Enterprise.",
     "relevant_ids": ["sso-01"]},
    {"question": "Co się stanie, jeśli moja integracja wyśle za dużo zapytań do API w planie Pro?",
     "reference": "Powyżej 600 zapytań na minutę API zwraca HTTP 429 z nagłówkiem Retry-After.",
     "relevant_ids": ["api-01"]},
    {"question": "Gdzie są przechowywane moje dane i jak są chronione?",
     "reference": "We Frankfurcie (UE) lub w Wirginii (USA), wybór przy tworzeniu przestrzeni; szyfrowanie AES-256 w spoczynku i TLS 1.3 w transmisji.",
     "relevant_ids": ["security-01"]},
    {"question": "Czy administrator może wymusić 2FA dla wszystkich członków?",
     "reference": "Tak, administratorzy planów Pro i Enterprise mogą wymusić 2FA (aplikacja uwierzytelniająca lub klucz sprzętowy).",
     "relevant_ids": ["security-02"]},
    {"question": "Jak pobrać wszystkie dane z mojej przestrzeni roboczej?",
     "reference": "Administrator zleca pełny eksport przestrzeni jako ZIP; jest gotowy w ciągu 24 godzin, a link jest ważny 7 dni.",
     "relevant_ids": ["export-01"]},
    {"question": "Jak szybko wsparcie odpowiada klientom planu Pro?",
     "reference": "E-mailem, z pierwszą odpowiedzią w ciągu 24 godzin roboczych.",
     "relevant_ids": ["support-01"]},
    {"question": "Usunąłem projekt dwa tygodnie temu. Czy mogę go jeszcze przywrócić?",
     "reference": "Tak, usunięte projekty są w Koszu przez 30 dni.",
     "relevant_ids": ["retention-01"]},
    {"question": "Gdzie utworzyć klucz API i jak długo jest ważny?",
     "reference": "W Ustawienia -> Deweloper; klucz wygasa po 365 dniach.",
     "relevant_ids": ["api-02"]},
    {"question": "Co się dzieje, gdy mój endpoint webhooka jest chwilowo niedostępny?",
     "reference": "Nieudane dostarczenia są ponawiane do 5 razy z wykładniczym opóźnieniem.",
     "relevant_ids": ["integrations-01"]},
    {"question": "Czy mogę zapłacić przelewem?",
     "reference": "Tylko w planie Enterprise, z 30-dniowym terminem płatności; plan Pro opłaca się kartą.",
     "relevant_ids": ["billing-01"]},
    {"question": "Potrzebujemy SSO i wsparcia telefonicznego 24/7. Który plan kupić?",
     "reference": "Enterprise - to jedyny plan z SSO i wsparciem telefonicznym 24/7.",
     "relevant_ids": ["sso-01", "support-01"]},
]
print(f"Liczba pytań w zbiorze testowym: {len(EVAL_DATASET)}")

Liczba pytań w zbiorze testowym: 15


### 3. Metryki wyszukiwania: Recall@K, Precision@K, MRR
To zwykłe funkcje na listach ID – nie potrzebują LLM, więc są szybkie, tanie i deterministyczne.

In [5]:
def recall_at_k(retrieved: list[str], relevant: list[str], k: int) -> float:
    """Jaka część istotnych dokumentów znalazła się w top-k wynikach."""
    return len(set(retrieved[:k]) & set(relevant)) / len(relevant)


def precision_at_k(retrieved: list[str], relevant: list[str], k: int) -> float:
    """Jaka część wyników top-k jest istotna."""
    return len(set(retrieved[:k]) & set(relevant)) / k


def reciprocal_rank(retrieved: list[str], relevant: list[str]) -> float:
    """1 / pozycja pierwszego trafnego wyniku (0, gdy brak trafienia). MRR = średnia po pytaniach."""
    for rank, doc_id in enumerate(retrieved, start=1):
        if doc_id in relevant:
            return 1 / rank
    return 0.0


# szybki test na prostym przykładzie
retrieved, relevant = ["a", "b", "c"], ["b", "d"]
assert recall_at_k(retrieved, relevant, k=3) == 0.5      # znaleziono b, pominięto d
assert precision_at_k(retrieved, relevant, k=3) == 1 / 3  # 1 z 3 wyników jest istotny
assert reciprocal_rank(retrieved, relevant) == 0.5       # pierwsze trafienie na pozycji 2
print("Metryki OK")

Metryki OK


#### Indeks wektorowy
Przy małym zbiorze wystarczy magazyn w pamięci; metryki działają tak samo z FAISS, Chroma czy pgvector.

In [6]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = InMemoryVectorStore.from_documents(docs, embedding=embeddings)


def retrieve_ids(question: str, k: int) -> list[str]:
    return [d.metadata["id"] for d in vectorstore.similarity_search(question, k=k)]


print(retrieve_ids("Czy moje dane są szyfrowane?", k=3))

['security-01', 'api-02', 'security-02']


#### Ewaluacja retrievera dla różnych K
Zwróć uwagę na kompromis: gdy K rośnie, **recall rośnie** (większa szansa na znalezienie dokumentu), ale **precision spada** (więcej szumu w kontekście, więcej tokenów i większe ryzyko rozproszenia modelu).

In [7]:
import pandas as pd

# Wyszukiwanie nie wywołuje LLM, więc pobieramy top 5 raz i wycinamy z niego wyniki dla każdego k.
retrieval_rows = []
for item in EVAL_DATASET:
    top5 = retrieve_ids(item["question"], k=5)
    row = {"question": item["question"], "relevant": item["relevant_ids"], "retrieved": top5,
           "MRR": reciprocal_rank(top5, item["relevant_ids"])}
    for k in (1, 3, 5):
        row[f"Recall@{k}"] = recall_at_k(top5, item["relevant_ids"], k)
        row[f"Precision@{k}"] = precision_at_k(top5, item["relevant_ids"], k)
    retrieval_rows.append(row)

retrieval_df = pd.DataFrame(retrieval_rows)
metric_cols = ["Recall@1", "Recall@3", "Recall@5", "Precision@1", "Precision@3", "Precision@5", "MRR"]
retrieval_df[metric_cols].mean().round(3).to_frame("średnia")

,średnia
Recall@1,0.833
Recall@3,0.967
Recall@5,0.967
Precision@1,0.867
Precision@3,0.356
Precision@5,0.213
MRR,0.933


In [8]:
# Pytania, w których pierwsze trafienie nie było na pozycji 1 - warto je przejrzeć ręcznie
retrieval_df.loc[retrieval_df["MRR"] < 1, ["question", "relevant", "retrieved", "MRR"]]

,question,relevant,retrieved,MRR
0,Ile kosztuje plan Pro?,[plans-01],"[trial-01, plans-01, billing-01, support-01, b...",0.5
3,Co się stanie z moją przestrzenią roboczą po z...,"[trial-01, plans-01]","[retention-01, trial-01, export-01, api-02, ap...",0.5


### 4. Pipeline RAG z cytowaniami, pomiarem czasu i tokenów
Model zwraca ustrukturyzowaną odpowiedź (`with_structured_output`) z listą ID źródeł, więc trafność cytowań da się sprawdzić automatycznie. `get_usage_metadata_callback` sumuje tokeny wszystkich wywołań modelu czatu w bloku, a `time.perf_counter` mierzy opóźnienie.

In [9]:
import time

from langchain_core.callbacks import get_usage_metadata_callback
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

TOP_K = 3


class CitedAnswer(BaseModel):
    """Odpowiedź na pytanie użytkownika wraz z ID dokumentów, na których się opiera."""
    answer: str = Field(description="Odpowiedź wyłącznie na podstawie kontekstu. Jeśli kontekst nie zawiera odpowiedzi, powiedz, że nie wiesz.")
    sources: list[str] = Field(description="ID dokumentów z kontekstu, które potwierdzają odpowiedź, np. ['plans-01'].")


rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Jesteś asystentem centrum pomocy Orbitly. Odpowiadaj zwięźle, korzystając WYŁĄCZNIE z kontekstu. "
               "Każdy dokument kontekstu zaczyna się od ID w nawiasach kwadratowych - podaj ID, z których korzystasz.\n\n"
               "KONTEKST:\n{context}"),
    ("user", "{question}"),
])
rag_chain = rag_prompt | ChatOpenAI(model="gpt-6-luna").with_structured_output(CitedAnswer)


def format_docs(documents: list[Document]) -> str:
    return "\n\n".join(f"[{d.metadata['id']}] {d.page_content}" for d in documents)


def ask(question: str, k: int = TOP_K) -> dict:
    """Uruchamia pipeline RAG i mierzy opóźnienie oraz zużycie tokenów."""
    start = time.perf_counter()
    with get_usage_metadata_callback() as usage:  # zbiera usage_metadata każdego wywołania modelu czatu
        context_docs = vectorstore.similarity_search(question, k=k)
        result = rag_chain.invoke({"context": format_docs(context_docs), "question": question})
    return {
        "answer": result.answer,
        "cited": result.sources,
        "retrieved": [d.metadata["id"] for d in context_docs],
        "context": format_docs(context_docs),
        "latency_s": time.perf_counter() - start,
        "tokens": sum(u["total_tokens"] for u in usage.usage_metadata.values()),
    }


ask("Czy mogę zapłacić przelewem?")

{'answer': 'Tak, jeśli masz plan Enterprise — możesz płacić przelewem z 30-dniowym terminem płatności. Klienci planu Pro płacą kartą kredytową.',
 'cited': ['billing-01'],
 'retrieved': ['billing-01', 'billing-02', 'trial-01'],
 'context': '[billing-01] Faktury są wystawiane 1. dnia każdego miesiąca i wysyłane na adres e-mail do rozliczeń. Klienci planu Pro płacą kartą kredytową. Klienci Enterprise mogą też płacić przelewem z 30-dniowym terminem płatności.\n\n[billing-02] Za plany roczne można otrzymać zwrot pieniędzy w ciągu 14 dni od zakupu. Plany miesięczne nie podlegają zwrotowi, ale anulowanie planu miesięcznego wstrzymuje kolejne obciążenie.\n\n[trial-01] Wersja próbna planu Pro trwa 21 dni i nie wymaga karty kredytowej. Po jej zakończeniu przestrzeń robocza automatycznie przechodzi na plan Free, a dane ponad limity planu Free stają się tylko do odczytu.',
 'latency_s': 1.4708418489899486,
 'tokens': 551}

### 5. LLM jako sędzia: faithfulness i answer correctness
- **Faithfulness** porównuje odpowiedź z *pobranym kontekstem* – wykrywa halucynacje, nawet jeśli odpowiedź przypadkiem jest prawdziwa.
- **Answer correctness** porównuje odpowiedź z *wzorcem* – wykrywa błędne lub niepełne odpowiedzi, nawet jeśli są wierne złemu kontekstowi.

Sędzia podaje uzasadnienie *przed* werdyktem, co zwiększa wiarygodność oceny i ułatwia jej audyt. Używaj modelu co najmniej tak mocnego jak oceniany.

In [10]:
class Verdict(BaseModel):
    reasoning: str = Field(description="Krótkie uzasadnienie, napisane przed werdyktem.")
    passed: bool = Field(description="Ostateczny werdykt.")


judge = ChatOpenAI(model="gpt-6.1-terra").with_structured_output(Verdict)

faithfulness_prompt = ChatPromptTemplate.from_messages([
    ("system", "Sprawdzasz odpowiedzi systemu RAG pod kątem halucynacji. Odpowiedź przechodzi, jeśli KAŻDE "
               "stwierdzenie faktyczne jest poparte kontekstem. Odpowiedź, że model nie wie, również przechodzi."),
    ("user", "KONTEKST:\n{context}\n\nODPOWIEDŹ:\n{answer}"),
])

correctness_prompt = ChatPromptTemplate.from_messages([
    ("system", "Porównujesz odpowiedź z odpowiedzią wzorcową. Odpowiedź przechodzi, jeśli zawiera kluczowe fakty "
               "z wzorca i mu nie przeczy. Dodatkowe poprawne szczegóły są w porządku."),
    ("user", "PYTANIE:\n{question}\n\nWZORZEC:\n{reference}\n\nODPOWIEDŹ:\n{answer}"),
])

faithfulness_judge = faithfulness_prompt | judge
correctness_judge = correctness_prompt | judge

# test poprawności: sędzia musi wychwycić zmyślony fakt
faithfulness_judge.invoke({
    "context": "[plans-01] Pro: 49 zł za użytkownika miesięcznie.",
    "answer": "Pro kosztuje 49 zł za użytkownika miesięcznie i zawiera darmową domenę.",
})

Verdict(reasoning='Cena jest poparta kontekstem, ale brak w nim informacji o darmowej domenie.', passed=False)

### 6. Trafność cytowań
Porównujemy ID wskazane przez model z ID ze zbioru testowego – znowu zwykła funkcja, bez LLM.

In [11]:
def citation_scores(cited: list[str], relevant: list[str]) -> tuple[float, float]:
    """Precyzja cytowań (czy wskazane źródła są właściwe?) i kompletność (czy wskazano wszystkie potrzebne?)."""
    hits = len(set(cited) & set(relevant))
    precision = hits / len(set(cited)) if cited else 0.0
    return precision, hits / len(relevant)


assert citation_scores(["a", "b"], ["a"]) == (0.5, 1.0)
assert citation_scores([], ["a"]) == (0.0, 0.0)

### 7. Pełne uruchomienie ewaluacji

In [12]:
def evaluate_example(item: dict, k: int = TOP_K) -> dict:
    out = ask(item["question"], k=k)
    faithful = faithfulness_judge.invoke({"context": out["context"], "answer": out["answer"]})
    correct = correctness_judge.invoke({"question": item["question"], "reference": item["reference"],
                                        "answer": out["answer"]})
    cit_precision, cit_recall = citation_scores(out["cited"], item["relevant_ids"])
    return {
        "question": item["question"],
        "answer": out["answer"],
        "cited": out["cited"],
        "relevant": item["relevant_ids"],
        f"Recall@{k}": recall_at_k(out["retrieved"], item["relevant_ids"], k),
        f"Precision@{k}": precision_at_k(out["retrieved"], item["relevant_ids"], k),
        "MRR": reciprocal_rank(out["retrieved"], item["relevant_ids"]),
        "faithfulness": faithful.passed,
        "correctness": correct.passed,
        "citation_precision": cit_precision,
        "citation_recall": cit_recall,
        "latency_s": out["latency_s"],  # tylko pipeline RAG, bez wywołań sędziego
        "tokens": out["tokens"],
        "judge_notes": f"F: {faithful.reasoning} | C: {correct.reasoning}",
    }


results = pd.DataFrame([evaluate_example(item) for item in EVAL_DATASET])
results[["question", "answer", "cited", "faithfulness", "correctness", "latency_s", "tokens"]]

,question,answer,cited,faithfulness,correctness,latency_s,tokens
0,Ile kosztuje plan Pro?,Plan Pro kosztuje 49 zł za użytkownika miesięc...,[plans-01],True,True,1.270333,506
1,Czy mogę dostać zwrot pieniędzy za plan miesię...,"Nie, plany miesięczne nie podlegają zwrotowi. ...",[billing-02],True,True,1.356346,518
2,Jak długo trwa okres próbny i czy potrzebuję k...,Okres próbny planu Pro trwa 21 dni i nie wymag...,[trial-01],True,True,1.462679,517
3,Co się stanie z moją przestrzenią roboczą po z...,Po zakończeniu 21-dniowego okresu próbnego prz...,[trial-01],True,False,2.498492,546
4,Których dostawców tożsamości mogę użyć do SSO?,"Do SSO możesz użyć Okta, Microsoft Entra ID (A...",[sso-01],True,False,1.446528,544
5,"Co się stanie, jeśli moja integracja wyśle za ...",W planie Pro obowiązuje limit 600 zapytań na m...,[api-01],True,True,1.187910,539
6,Gdzie są przechowywane moje dane i jak są chro...,Dane są przechowywane we Frankfurcie (UE) albo...,[security-01],True,True,1.564134,557
7,Czy administrator może wymusić 2FA dla wszystk...,Tak. Administratorzy planów Pro i Enterprise m...,[security-02],True,True,1.330479,510
8,Jak pobrać wszystkie dane z mojej przestrzeni ...,Administrator może zlecić pełny eksport przest...,[export-01],True,True,1.261235,551
9,Jak szybko wsparcie odpowiada klientom planu Pro?,Użytkownicy planu Pro otrzymują pierwszą odpow...,[support-01],True,False,1.126310,519


#### Raport zbiorczy

In [13]:
report = {
    f"Recall@{TOP_K}": results[f"Recall@{TOP_K}"].mean(),
    f"Precision@{TOP_K}": results[f"Precision@{TOP_K}"].mean(),
    "MRR": results["MRR"].mean(),
    "Faithfulness": results["faithfulness"].mean(),
    "Answer correctness": results["correctness"].mean(),
    "Citation precision": results["citation_precision"].mean(),
    "Citation recall": results["citation_recall"].mean(),
    "Latency p50 [s]": results["latency_s"].median(),
    "Latency p95 [s]": results["latency_s"].quantile(0.95),
    "Tokeny / pytanie": results["tokens"].mean(),
    "Tokeny łącznie": results["tokens"].sum(),
}
pd.Series(report).round(3).to_frame("wartość")

,wartość
Recall@3,0.967
Precision@3,0.356
MRR,0.933
Faithfulness,0.933
Answer correctness,0.800
Citation precision,0.978
Citation recall,0.967
Latency p50 [s],1.356
Latency p95 [s],1.844
Tokeny / pytanie,531.933


In [14]:
# Przykłady nieudane lub niedoskonałe - najcenniejsza część raportu
failed = results[~results["faithfulness"] | ~results["correctness"] | (results["citation_precision"] < 1)]
print(f"Przykłady do przejrzenia: {len(failed)}")
for _, row in failed.iterrows():
    print(f"\nP: {row['question']}\nO: {row['answer']}\ncytowane={row['cited']} istotne={row['relevant']}\n{row['judge_notes']}")

Przykłady do przejrzenia: 5

P: Co się stanie z moją przestrzenią roboczą po zakończeniu okresu próbnego?
O: Po zakończeniu 21-dniowego okresu próbnego przestrzeń robocza automatycznie przejdzie na plan Free. Dane przekraczające limity tego planu będą tylko do odczytu.
cytowane=['trial-01'] istotne=['trial-01', 'plans-01']
F: Oba stwierdzenia są bezpośrednio poparte fragmentem trial-01. | C: Odpowiedź poprawnie wskazuje przejście na plan Free i tryb tylko do odczytu, ale pomija jego limity: 3 projekty i 1 GB.

P: Których dostawców tożsamości mogę użyć do SSO?
O: Do SSO możesz użyć Okta, Microsoft Entra ID (Azure AD) lub Google Workspace. SSO jest dostępne wyłącznie w planie Enterprise.
cytowane=['sso-01'] istotne=['sso-01']
F: Kontekst potwierdza wszystkich wymienionych dostawców SSO oraz dostępność SSO wyłącznie w planie Enterprise. | C: Odpowiedź wymienia właściwych dostawców i plan Enterprise, ale pomija obsługiwane protokoły SAML 2.0 i OIDC.

P: Jak szybko wsparcie odpowiada klient

### 8. Bramka jakości
Same średnie nie mówią, czy wdrażać zmianę. Ustal progi i sprawdzaj je po każdej modyfikacji pipeline'u.

Gdy metryka nie przechodzi, najpierw przeczytaj nieudane przykłady: czasem błąd jest w pipelinie, a czasem wzorzec lub sędzia są zbyt surowi (np. wzorzec zawiera szczegóły, o które pytanie nie prosiło). Poprawiaj zbiór testowy równie starannie jak kod.

In [15]:
# Progi uzgodnione z zespołem - test regresji uruchamiany po każdej zmianie
# (nowy prompt, model, chunking, embeddingi...). W CI zamień print na assert.
THRESHOLDS = {f"Recall@{TOP_K}": 0.9, "Faithfulness": 0.9, "Answer correctness": 0.85,
              "Citation precision": 0.8, "Latency p95 [s]": 10.0}

for name, threshold in THRESHOLDS.items():
    value = report[name]
    ok = value <= threshold if name.startswith("Latency") else value >= threshold
    print(f"{'PASS' if ok else 'FAIL'}  {name}: {value:.3f} (próg {threshold})")

PASS  Recall@3: 0.967 (próg 0.9)
PASS  Faithfulness: 0.933 (próg 0.9)
FAIL  Answer correctness: 0.800 (próg 0.85)
PASS  Citation precision: 0.978 (próg 0.8)
PASS  Latency p95 [s]: 1.844 (próg 10.0)


### Podsumowanie i kolejne kroki
- Oceniaj **wyszukiwanie i generowanie osobno** – to mówi, *co* poprawić: chunking/embeddingi/K czy prompt/model.
- Trzymaj zbiór testowy w repozytorium i **rozszerzaj go o każdy błąd** zgłoszony przez użytkowników.
- Sędziowie LLM nie są nieomylni – co jakiś czas przejrzyj ręcznie próbkę ich werdyktów.
- Aby śledzić eksperymenty w czasie, wgraj zbiór do LangSmith i uruchom te same ewaluatory przez `langsmith.evaluate` (zob. notebook 6_3). Biblioteki takie jak RAGAS oferują gotowe wersje tych metryk.